# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Sinh viên:** Nguyễn Xuân Thành  
**MSSV:** 2A202602666  
**Môn học:** VIN AI Thực Chiến — Track 4 Day 2 (Deep Learning Advance)

Notebook này hoàn thiện toàn bộ các bước thực nghiệm theo đúng quy định của `README.md`, `GUIDE.md` và `RUBRIC.md`.

## Cài đặt thư viện và môi trường

In [ ]:
import os, sys, platform
from pathlib import Path

# Đảm bảo UTF-8 cho Windows console và file I/O
if sys.platform == 'win32':
    if hasattr(sys.stdout, 'reconfigure'):
        sys.stdout.reconfigure(encoding='utf-8')
    if hasattr(sys.stderr, 'reconfigure'):
        sys.stderr.reconfigure(encoding='utf-8')

# Tự động thiết lập đường dẫn thư mục gốc và mã nguồn
CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR
for p in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (p / 'README.md').exists() and (p / 'data').exists():
        REPO_ROOT = p
        break
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
code_dir = REPO_ROOT / 'submissions' / '2A202602666_NguyenXuanThanh' / 'code'
if code_dir.exists() and str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))
elif (REPO_ROOT / 'starter').exists() and str(REPO_ROOT / 'starter') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'starter'))

try:
    import torch, timm
    print('[OK] Môi trường Python sẵn sàng!')
    print('Python Executable:', sys.executable)
    print(f'Python: {platform.python_version()} | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
    print('timm version:', timm.__version__)
except ImportError as e:
    print('[!] LỖI THIẾU THƯ VIỆN:', e)
    print('Notebook đang chạy trên Python interpreter:', sys.executable)
    print('-> Hãy chọn lại Kernel ở góc trên bên phải sang: Python 3.10 (with PyTorch GPU)')
    raise


## Bước 0 — EDA và kiểm tra tính toàn vẹn dữ liệu (S1 - S6)

In [ ]:
import sys, os
from pathlib import Path

# Tự động thêm đường dẫn code vào sys.path để không bao giờ bị lỗi ModuleNotFoundError: dataset
for p in [Path.cwd() / 'code', Path.cwd() / 'submissions/2A202602666_NguyenXuanThanh/code', Path.cwd()]:
    if p.exists() and (p / 'dataset.py').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dataset as ds
import pandas as pd
import matplotlib.pyplot as plt

IMAGES_DIR = 'data/images'
LABELS_DIR = 'data/labels'

# 1. Đọc và kiểm tra chia tập dữ liệu Fold 0
train_df, val_df, test_df = ds.load_split(LABELS_DIR, fold=0)
split_info = ds.check_split(train_df, val_df, test_df, IMAGES_DIR)

print(f"Tổng số mẫu: {split_info['total']}")
print(f"Train: {split_info['n_train']} ({split_info['train_pct']:.2%})")
print(f"Val:   {split_info['n_val']} ({split_info['val_pct']:.2%})")
print(f"Test:  {split_info['n_test']} ({split_info['test_pct']:.2%})")
print('Độ giao thoa giữa các tập (overlap):', split_info['overlap_tv'], split_info['overlap_tt'], split_info['overlap_vt'])

# 2. Vẽ biểu đồ phân bố 9 lớp
counts = train_df['Label'].value_counts().sort_index()
plt.figure(figsize=(10, 4), dpi=150)
plt.bar([ds.CLASS_NAMES[i] for i in counts.index], counts.values, color='#1F77B4', edgecolor='black')
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.ylabel('Số lượng ảnh (Train)', fontsize=11)
plt.title('Phân bố các lớp dữ liệu DeepWeeds (Fold 0 Train)', fontsize=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### Kiểm tra Pipeline trước khi huấn luyện (Sanity Checks)
1. Cố định hạt giống ngẫu nhiên (seed)
2. Kiểm tra loss ban đầu xấp xỉ `-ln(1/9) ≈ 2.197`
3. Quá khớp (overfit) một batch nhỏ tới loss gần 0
4. Kiểm tra Focal Loss khi `gamma=0` bằng chính xác CrossEntropy

In [ ]:
import math
import torch.nn as nn
import model as md
import losses as ls
import train

# 1. Seed
train.set_seed(42)

# 2. Initial Loss Check (dùng pretrained=False để kiểm tra phân bố ngẫu nhiên lý thuyết)
dummy_model = md.build_model('resnet50', pretrained=False, num_classes=9)
criterion = nn.CrossEntropyLoss()
dummy_x = torch.randn(8, 3, 224, 224)
dummy_y = torch.randint(0, 9, (8,))
with torch.no_grad():
    init_logits = dummy_model(dummy_x)
    init_loss = criterion(init_logits, dummy_y).item()
expected_loss = -math.log(1.0 / 9.0)
print(f'Initial Loss: {init_loss:.4f} | Lý thuyết -ln(1/9): {expected_loss:.4f}')

# 3. Focal Loss gamma=0 == CE check
fl_0 = ls.FocalLoss(gamma=0.0)(init_logits, dummy_y).item()
diff = abs(fl_0 - init_loss)
print(f'Focal Loss (gamma=0) diff from CE: {diff:.8f} (< 1e-6: ĐẠT)')
assert diff < 1e-6


## Bước 1 — So sánh các Backbone (≥ 5 Backbone)
So sánh công bằng trên công thức nền `T00` (12 epochs, AdamW, cosine schedule, basic augmentation):
- `resnet50`: Mốc chuẩn CNN cổ điển
- `resnext50_32x4d`: Mạng CNN mở rộng cardinality
- `convnext_tiny`: Mạng CNN hiện đại hoá (Modernized CNN)
- `deit_small_patch16_224`: Vision Transformer
- `efficientnet_b0`: Mạng CNN nhẹ hiệu năng cao
- `mobilenetv3_large_100`: Mạng CNN siêu nhẹ cho vi xử lý nhúng

In [ ]:
backbones_list = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "efficientnet_b0"),
    ("B06", "mobilenetv3_large_100"),
]

print("Danh sách backbone tham gia so sánh:")
for exp_id, b_name in backbones_list:
    m = md.build_model(b_name, pretrained=False)
    p = md.count_params(m)
    g = md.count_gmacs(m)
    print(f"[{exp_id}] {b_name:<28} | Params: {p:>6.2f}M | GMACs: {g:>5.2f}")

# Ví dụ lệnh chạy thực nghiệm (huấn luyện 1 backbone):
# cfg = train.Config(exp_id='B01', backbone='resnet50', epochs=12, seed=0)
# res = train.run(cfg)

## Bước 2 — Khảo sát công thức huấn luyện (Training Recipes Ablation)
Khảo sát có kiểm soát từng trục (mỗi lần chỉ đổi đúng 1 yếu tố so với mốc `T00`):
- **Trục A (Khởi tạo):** From scratch (`T01`), Frozen linear probe (`T02`)
- **Trục B (Augmentation):** Color jitter (`T03`), RandAugment (`T04`), CutMix (`T05`)
- **Trục C (Loss function):** Label Smoothing (`T06`), Focal Loss (`T07`), Class-Weighted CE (`T08`)
- **Trục D (Sampling):** Balanced Weighted Random Sampler (`T09`)
- **Trục F (Regularization):** Weight Exponential Moving Average (`T10`)
- **Tổ hợp tốt nhất:** `convnext_tiny` + CutMix + Label Smoothing + EMA (`T11`)

In [ ]:
# Ví dụ cấu hình cho thí nghiệm kết hợp tối ưu T11:
cfg_t11 = train.Config(
    exp_id="T11",
    backbone="convnext_tiny",
    aug="basic",
    mix="cutmix",
    mix_alpha=1.0,
    loss="ls",
    label_smoothing=0.1,
    ema_decay=0.999,
    epochs=12,
    seed=0
)
print("Cấu hình tối ưu T11 đã sẵn sàng:", cfg_t11.exp_id, cfg_t11.backbone)

## Bước 3 — Phương pháp suy luận và Đo lường độ trễ chuẩn GPU
Khảo sát các kỹ thuật suy luận không cần huấn luyện lại:
- `I00`: 1 view chuẩn (mốc)
- `I01`: TTA lật ngang (K=2)
- `I02`: TTA 5-crop (K=5)
- `I05`: Ensemble đa mô hình qua các seed
- `I07`: Hiệu chuẩn xác suất Temperature Scaling (khớp T trên Val, đo ECE trước và sau)
- `I08`: Gộp BatchNorm vào Conv2d để tăng tốc suy luận

In [ ]:
import benchmark as bm

# Đo độ trễ chuẩn forward batch 1 và batch 32 trên GPU
bench_model = md.build_model("resnet50", pretrained=False)
device = "cuda" if torch.cuda.is_available() else "cpu"

lat_b1 = bm.latency_report(bench_model, batch_size=1, img_size=224, dtype="fp32", device=device, warmup=10, iters=50)
print("Độ trễ Batch 1 (Robot Real-time):")
print(f"  GPU: {lat_b1['gpu']} | p50: {lat_b1['p50']} ms | p95: {lat_b1['p95']} ms | p99: {lat_b1['p99']} ms | FPS: {lat_b1['images_per_s']}")

lat_b32 = bm.latency_report(bench_model, batch_size=32, img_size=224, dtype="fp32", device=device, warmup=5, iters=20)
print("Thông lượng Batch 32:")
print(f"  p50: {lat_b32['p50']} ms | Thông lượng: {lat_b32['images_per_s']} ảnh/giây")

## Bước 4 — Chung kết: Đánh giá ≥ 3 Seed trên Test và Chấm điểm tự động
Chạy cấu hình chung kết `F01` và mốc `T00` trên 3 seed (0, 1, 2) độc lập.
Chạy công cụ `eval.py score` và `eval.py grade` để tự chấm phần I của RUBRIC.

In [ ]:
import subprocess, sys

cmd_score = [
    sys.executable, 'eval.py', 'score',
    '--pred', 'submissions/2A202602666_NguyenXuanThanh/predictions/F01_seed*_test.csv',
    '--test-csv', 'data/labels/test_subset0.csv',
    '--labels', 'data/labels/labels.csv',
    '--tag', 'F01',
    '--out', 'submissions/2A202602666_NguyenXuanThanh/eval_out'
]
print('=== BẮT ĐẦU CHẠY EVAL SCORE ===')
subprocess.run(cmd_score, check=True)

cmd_grade = [
    sys.executable, 'eval.py', 'grade',
    '--final', 'submissions/2A202602666_NguyenXuanThanh/predictions/F01_seed*_test.csv',
    '--baseline', 'submissions/2A202602666_NguyenXuanThanh/predictions/T00_seed*_test.csv',
    '--uncal', 'submissions/2A202602666_NguyenXuanThanh/predictions/F01uncal_seed*_test.csv',
    '--final-val', 'submissions/2A202602666_NguyenXuanThanh/predictions/F01_seed*_val.csv',
    '--latency-p95-ms', '17.61',
    '--latency-method', 'proper',
    '--test-csv', 'data/labels/test_subset0.csv',
    '--labels', 'data/labels/labels.csv',
    '--out', 'submissions/2A202602666_NguyenXuanThanh/eval_out'
]
print('\n=== BẮT ĐẦU CHẠY EVAL GRADE ===')
subprocess.run(cmd_grade, check=True)


## Bước 5 — Tổng kết sản phẩm
Kiểm tra file kết quả `results.xlsx`, các biểu đồ trong `curves/`, dự đoán trong `predictions/` và báo cáo `report.md`.

In [ ]:
import openpyxl
wb = openpyxl.load_workbook("submissions/2A202602666_NguyenXuanThanh/results.xlsx")
print("Các sheet có trong results.xlsx:", wb.sheetnames)
ws_summary = wb["Summary"]
for row in list(ws_summary.iter_rows(values_only=True))[:6]:
    print(row)